# zhanfa 快速上手

完整演示：**获取数据 → 清洗 → 策略信号 → 回测 → 绩效报告**

## 安装初始化

首次使用时，从聚宽获取所有A股列表，并设置基准。
```python
# 可选，提前将成分股列表缓存到本地
from zhanfa.data.fetcher import Fetcher
fetcher = Fetcher()
codes = fetcher.index_components("000300")
fetcher.daily_batch(codes)  # 批量拉取并缓存所有沪深300日线数据
```

数据缓存自动管理，只需运行一次，后续加载即用本地数据，速度快且不受API限制。

In [ ]:
import pandas as pd
import numpy as np

from zhanfa.data.fetcher import Fetcher
from zhanfa.data.pipeline import Pipeline
from zhanfa.strategies.trend.sma_cross import SMACross
from zhanfa.strategies.trend.turtle import Turtle
from zhanfa.backtest.engine import run_backtest_from_strategy, compare_strategies
from zhanfa.backtest.metrics import compute_metrics
from zhanfa.backtest.report import generate_report

pd.set_option('display.max_rows', 8)
pd.set_option('display.expand_frame_repr', False)

## 1. 获取数据

In [ ]:
fetcher = Fetcher()

# 沪深300指数日线（首次从 akshare 拉取，之后走本地缓存）
df = fetcher.index_daily('000300')
print(f"数据范围: {df.index[0].date()} ~ {df.index[-1].date()}")
print(f"共 {len(df)} 天")
df.head()

In [ ]:
df[['close']].plot(figsize=(14, 5), title='沪深300 收盘价', grid=True);

## 2. 清洗 + 特征计算

In [ ]:
df = Pipeline.clean(df)
df = Pipeline.add_simple_indicators(df)
print(f"{len(df)} 行, {len(df.columns)} 列")
df[['close', 'sma_20', 'sma_60', 'atr_14', 'volatility_20d']].tail()

## 3. 策略信号

In [ ]:
sma = SMACross(fast=20, slow=60)
signals = sma.generate_signals(df)

# 买入信号标记
buy_points = signals[signals & ~signals.shift(1).fillna(False)]
sell_points = (~signals)[signals.shift(1).fillna(False)]

ax = df['close'].plot(figsize=(14, 5), alpha=0.6, label='close', title='双均线交叉(20/60)')
ax.scatter(buy_points.index, df.loc[buy_points.index, 'close'], marker='^', c='g', s=50, label='buy')
ax.scatter(sell_points.index, df.loc[sell_points.index, 'close'], marker='v', c='r', s=50, label='sell')
ax.legend();

## 4. 回测

In [ ]:
pf = run_backtest_from_strategy(df, sma)
pf.value().plot(figsize=(14, 5), title='双均线策略权益曲线', grid=True);

## 5. 绩效报告

In [ ]:
print(generate_report(pf))

In [ ]:
metrics = compute_metrics(pf.value())
for k, v in metrics.items():
    label = {
        'total_return': '总收益率', 'ann_return': '年化收益率',
        'ann_volatility': '年化波动率', 'sharpe': '夏普比率',
        'sortino': '索提诺比率', 'max_drawdown': '最大回撤',
        'calmar': '卡玛比率', 'win_rate': '胜率', 'years': '回测年数'
    }.get(k, k)
    if isinstance(v, float):
        fmt = f"{v:.4f}" if abs(v) < 1 else f"{v:.2%}"
        print(f"{label:12s}: {fmt}")
    else:
        print(f"{label:12s}: {v}")

## 6. 多策略对比

In [ ]:
price = df['close']
results = compare_strategies(price, [
    SMACross(fast=5, slow=20),
    SMACross(fast=20, slow=60),
    SMACross(fast=10, slow=50),
    Turtle(),
])
results

In [ ]:
# 多策略权益曲线对比
equity_curves = {}
for s in [SMACross(fast=20, slow=60), Turtle()]:
    pf = run_backtest_from_strategy(price, s)
    equity_curves[s.name] = pf.value()

pd.DataFrame(equity_curves).plot(figsize=(14, 5), title='权益曲线对比', grid=True);

## 7. 回撤分析

In [ ]:
dd = pf.drawdown()
dd.plot(figsize=(14, 4), title='策略回撤', grid=True);

## 8. 导出 JoinQuant

In [ ]:
from zhanfa.jq.adapter import to_jq_template

code = to_jq_template(SMACross(fast=20, slow=60))
print(code)